# Continuous phylogeography and incidence

This notebook visualizes the 2026 BEAST continuous diffusion summaries and evaluates lagged Spearman cross-correlation with updated global monthly measles incidence.

Kept outputs:
- Diffusion velocity and diffusion coefficient with global incidence overlay
- Insets showing posterior run traces, incidence, and two predefined epochs
- Cross-correlation curves for the same two parameters
- Correlation summary table

In [ ]:

from pathlib import Path
from datetime import datetime
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from scipy.stats import spearmanr, pearsonr
from IPython.display import display

warnings.filterwarnings("ignore", category=RuntimeWarning)

BASE_DIR = Path(".")
LOG_PATH = BASE_DIR / "geo.continuous.combined.resamp50000.log"
INCIDENCE_PATH = BASE_DIR / "case_status_pop_incidence_2011_2026_updated.csv"

ABS_TIME_STR = "2026-05-28"
BURNIN_FRACTION = 0.0
LAG_RANGE_MONTHS = range(-24, 25)
CORRELATION_METHOD = "spearman"

FIG_DIFFUSION_PDF = BASE_DIR / "continuous_diffusion_velocity_coefficient_with_incidence.pdf"
FIG_DIFFUSION_PNG = BASE_DIR / "continuous_diffusion_velocity_coefficient_with_incidence.preview.png"
FIG_XCORR_PDF = BASE_DIR / "continuous_diffusion_velocity_coefficient_cross_correlation.pdf"
FIG_XCORR_PNG = BASE_DIR / "continuous_diffusion_velocity_coefficient_cross_correlation.preview.png"
SUMMARY_CSV = BASE_DIR / "continuous_diffusion_velocity_coefficient_cross_correlation_summary.csv"

EPOCHS = [
    (2018.2, 2020.2, "#E9EAF5"),
    (2020.3, 2023.4, "#F8E1E1"),
]

PARAMS = [
    {
        "prefix": "location.diffusionRate.height",
        "key": "diffusion_velocity",
        "title": "Diffusion velocity",
        "ylabel": "Diffusion velocity (km/year)",
        "line_color": "#9B3557",
    },
    {
        "prefix": "location.diffusionCoefficientWA.height",
        "key": "diffusion_coefficient",
        "title": "Diffusion coefficient",
        "ylabel": "Diffusion coefficient (km^2/year)",
        "line_color": "#7A4D8F",
    },
]

INCIDENCE_COLOR = "#6F927F"
TRACE_COLOR = "#C7CCD1"

mpl.rcParams.update({
    "text.usetex": False,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "Microsoft YaHei", "SimHei"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "axes.unicode_minus": False,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
})


In [ ]:

def decimal_year(date_string):
    dt = datetime.strptime(date_string, "%Y-%m-%d")
    start = datetime(dt.year, 1, 1)
    return dt.year + (dt - start).days / 365.25


def month_decimal_year(dates):
    dates = pd.to_datetime(dates)
    return dates.dt.year + (dates.dt.month - 0.5) / 12.0


def read_beast_log(path, burnin_fraction=0.0):
    df = pd.read_csv(path, sep=r"\s+", comment="#", engine="python")
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="ignore")
    if burnin_fraction and burnin_fraction > 0:
        if "state" in df.columns:
            cutoff = df["state"].quantile(burnin_fraction)
            df = df.loc[df["state"] >= cutoff].reset_index(drop=True)
        else:
            df = df.iloc[int(len(df) * burnin_fraction):].reset_index(drop=True)
    return df


def extract_height_matrix(df, prefix, abs_time):
    pat = re.compile(rf"^{re.escape(prefix)}([0-9]+(?:\.[0-9]+)?)$")
    hits = []
    for col in df.columns:
        m = pat.match(col)
        if m:
            hits.append((float(m.group(1)), col))
    if not hits:
        raise ValueError(f"No columns found for prefix: {prefix}")

    hits.sort(key=lambda item: item[0])
    heights = np.array([height for height, _ in hits], dtype=float)
    time = abs_time - heights
    mat = np.column_stack([pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=float) for _, col in hits])

    order = np.argsort(time)
    return time[order], mat[:, order]


def summarize_matrix(time, mat):
    return pd.DataFrame({
        "time": time,
        "median": np.nanmedian(mat, axis=0),
        "q25": np.nanquantile(mat, 0.25, axis=0),
        "q75": np.nanquantile(mat, 0.75, axis=0),
    })


def read_global_incidence(path):
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["time"], format="%Y-%m")
    monthly = (
        df.groupby("Date", as_index=False)
          .agg(cases=("cases", "sum"), population=("population", "sum"))
          .sort_values("Date")
    )
    monthly["incidence_per_100k"] = monthly["cases"] / monthly["population"] * 100000.0
    monthly["time_num"] = month_decimal_year(monthly["Date"])
    return monthly


abs_time = decimal_year(ABS_TIME_STR)
log_df = read_beast_log(LOG_PATH, BURNIN_FRACTION)
incidence_df = read_global_incidence(INCIDENCE_PATH)

param_data = {}
for item in PARAMS:
    time, mat = extract_height_matrix(log_df, item["prefix"], abs_time)
    summary = summarize_matrix(time, mat)
    param_data[item["key"]] = {"time": time, "mat": mat, "summary": summary, **item}

print(f"Loaded BEAST log: {log_df.shape[0]} posterior rows, {log_df.shape[1]} columns")
print(f"Loaded incidence: {incidence_df['Date'].min().date()} to {incidence_df['Date'].max().date()} ({len(incidence_df)} months)")
print(f"Absolute time used for height conversion: {ABS_TIME_STR} = {abs_time:.4f}")
display(incidence_df.tail())


In [ ]:

def setup_common_axis(ax, x_min, x_max):
    ax.set_xlim(x_min, x_max)
    ax.grid(axis="y", color="#E3E6EA", lw=0.55, alpha=0.9)
    ax.grid(axis="x", color="#F0F2F4", lw=0.45, alpha=0.58)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    ax.spines["left"].set_color("#303030")
    ax.spines["bottom"].set_color("#303030")
    ax.tick_params(axis="both", labelsize=10.5, colors="#303030", length=4)


def add_epochs(ax, alpha=0.74, zorder=0):
    for start, end, color in EPOCHS:
        ax.axvspan(start, end, color=color, alpha=alpha, lw=0, zorder=zorder)


def add_trace_inset(ax, data, incidence_df, x_min, x_max):
    inset = ax.inset_axes([0.075, 0.61, 0.34, 0.34])
    inset_inc = inset.twinx()

    add_epochs(inset, alpha=0.84, zorder=0)

    inc_time = incidence_df["time_num"].to_numpy(dtype=float)
    inc_vals = incidence_df["incidence_per_100k"].to_numpy(dtype=float)
    inset_inc.bar(inc_time, inc_vals, width=0.07, color=INCIDENCE_COLOR, alpha=0.22, edgecolor="none", zorder=1)
    inset_inc.set_ylim(0, np.nanmax(inc_vals) * 1.12)
    inset_inc.tick_params(axis="y", labelsize=4.8, colors="#557365", length=1.8, width=0.45, pad=1)
    inset_inc.spines["top"].set_visible(False)
    inset_inc.spines["left"].set_visible(False)
    inset_inc.spines["right"].set_color("#9BAEA4")
    inset_inc.spines["right"].set_linewidth(0.45)

    time = data["time"]
    mat = data["mat"]
    for row in mat:
        ok = np.isfinite(time) & np.isfinite(row)
        if ok.sum() >= 2:
            inset.plot(time[ok], row[ok], color=TRACE_COLOR, alpha=0.045, lw=0.45, zorder=2)

    s = data["summary"]
    inset.plot(s["time"], s["median"], color=data["line_color"], lw=0.95, zorder=4)
    inset.set_xlim(x_min, x_max)
    inset.set_title(data["title"], fontsize=5.8, pad=1.5)
    inset.tick_params(axis="both", labelsize=4.8, colors="#3B3B3B", length=1.8, width=0.45, pad=1)
    inset.grid(axis="y", color="#E8EAED", lw=0.35, alpha=0.8)
    inset.grid(axis="x", color="#F1F2F4", lw=0.25, alpha=0.55)
    for side in ["top", "right"]:
        inset.spines[side].set_visible(False)
    inset.spines["left"].set_linewidth(0.5)
    inset.spines["bottom"].set_linewidth(0.5)
    inset.spines["left"].set_color("#333333")
    inset.spines["bottom"].set_color("#333333")

    if data["key"] == "diffusion_coefficient":
        fmt = ScalarFormatter(useMathText=True)
        fmt.set_powerlimits((0, 0))
        inset.yaxis.set_major_formatter(fmt)
        inset.yaxis.offsetText.set_fontsize(4.8)

    return inset


def plot_diffusion_with_incidence(param_data, incidence_df):
    x_min = 2011.0
    x_max = 2026.55
    inc_time = incidence_df["time_num"].to_numpy(dtype=float)
    inc_vals = incidence_df["incidence_per_100k"].to_numpy(dtype=float)

    fig, axes = plt.subplots(1, 2, figsize=(13.6, 5.3), sharex=True, constrained_layout=False)
    fig.patch.set_facecolor("white")

    for idx, (ax, item) in enumerate(zip(axes, PARAMS)):
        data = param_data[item["key"]]
        s = data["summary"]
        color = item["line_color"]

        ax_inc = ax.twinx()
        ax.set_zorder(2)
        ax.patch.set_visible(False)
        ax_inc.set_zorder(1)
        ax_inc.bar(inc_time, inc_vals, width=0.075, color=INCIDENCE_COLOR, alpha=0.27, edgecolor="none", zorder=0)
        ax_inc.set_ylim(0, np.nanmax(inc_vals) * 1.18)
        ax_inc.tick_params(axis="y", labelsize=10.0, colors="#6E897A", length=3.4, width=0.75)
        ax_inc.spines["top"].set_visible(False)
        ax_inc.spines["left"].set_visible(False)
        ax_inc.spines["right"].set_color("#9BAEA4")
        ax_inc.spines["right"].set_linewidth(0.8)
        if idx == 0:
            ax_inc.set_yticklabels([])
            ax_inc.set_ylabel("")
        else:
            ax_inc.set_ylabel("Global incidence (cases per 100,000)", fontsize=11.5, color="#6E897A", labelpad=10)

        ax.fill_between(s["time"], s["q25"], s["q75"], color=color, alpha=0.23, lw=0, label="50% CrI", zorder=3)
        ax.plot(s["time"], s["median"], color=color, lw=2.1, solid_capstyle="round", label="Median", zorder=4)

        ax.set_title(item["title"], fontsize=22, pad=14, color="#111111")
        ax.set_ylabel(item["ylabel"], fontsize=12.5, labelpad=8)
        setup_common_axis(ax, x_min, x_max)

        if item["key"] == "diffusion_coefficient":
            fmt = ScalarFormatter(useMathText=True)
            fmt.set_powerlimits((0, 0))
            ax.yaxis.set_major_formatter(fmt)
            ax.yaxis.offsetText.set_fontsize(9)

        add_trace_inset(ax, data, incidence_df, x_min, x_max)

    years = np.arange(2011, 2027, 2)
    for ax in axes:
        ax.set_xticks(years)
        ax.set_xticklabels([str(y) for y in years], fontsize=10.5)
        ax.set_xlabel("Year", fontsize=11.5)

    incidence_patch = mpl.patches.Patch(facecolor=INCIDENCE_COLOR, alpha=0.27, edgecolor="none", label="Global incidence")
    axes[1].legend(
        handles=[incidence_patch],
        loc="upper right",
        bbox_to_anchor=(0.965, 1.025),
        frameon=False,
        fontsize=11.5,
        handlelength=1.6,
        borderaxespad=0.0,
    )

    plt.subplots_adjust(left=0.07, right=0.925, bottom=0.13, top=0.84, wspace=0.23)
    return fig


fig = plot_diffusion_with_incidence(param_data, incidence_df)
fig.savefig(FIG_DIFFUSION_PDF, format="pdf", bbox_inches="tight")
fig.savefig(FIG_DIFFUSION_PNG, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIFFUSION_PDF}")
print(f"Saved: {FIG_DIFFUSION_PNG}")


In [ ]:

def interpolate_matrix_to_target_time(src_time, mat, target_time):
    target_time = np.asarray(target_time, dtype=float)
    out = np.full((mat.shape[0], len(target_time)), np.nan)
    for i in range(mat.shape[0]):
        y = mat[i, :]
        ok = np.isfinite(src_time) & np.isfinite(y)
        if ok.sum() < 2:
            continue
        xv = src_time[ok]
        yv = y[ok]
        inside = (target_time >= xv.min()) & (target_time <= xv.max())
        out[i, inside] = np.interp(target_time[inside], xv, yv)
    return out


def align_with_lag(x, y, lag_months):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if lag_months > 0:
        x2, y2 = x[:-lag_months], y[lag_months:]
    elif lag_months < 0:
        k = abs(lag_months)
        x2, y2 = x[k:], y[:-k]
    else:
        x2, y2 = x.copy(), y.copy()
    ok = np.isfinite(x2) & np.isfinite(y2)
    return x2[ok], y2[ok]


def correlation_at_lag(x, y, lag_months=0, method="spearman"):
    x2, y2 = align_with_lag(x, y, lag_months)
    if len(x2) < 4:
        return np.nan, np.nan, len(x2)
    if method == "spearman":
        rho, p = spearmanr(x2, y2)
    elif method == "pearson":
        rho, p = pearsonr(x2, y2)
    else:
        raise ValueError("method must be 'spearman' or 'pearson'")
    return rho, p, len(x2)


def scan_lags(x, y, lag_range_months=LAG_RANGE_MONTHS, method=CORRELATION_METHOD):
    rows = []
    for lag in lag_range_months:
        rho, p, n = correlation_at_lag(x, y, lag_months=lag, method=method)
        rows.append({"lag_months": lag, "correlation": rho, "p_value": p, "n_pairs": n})
    return pd.DataFrame(rows)


def find_best_lag(lag_df, criterion="abs"):
    tmp = lag_df.dropna(subset=["correlation"]).copy()
    if tmp.empty:
        return None
    if criterion == "abs":
        idx = tmp["correlation"].abs().idxmax()
    elif criterion == "positive":
        idx = tmp["correlation"].idxmax()
    elif criterion == "negative":
        idx = tmp["correlation"].idxmin()
    else:
        raise ValueError("criterion must be 'abs', 'positive', or 'negative'")
    return tmp.loc[idx]


inc_time = incidence_df["time_num"].to_numpy(dtype=float)
inc_values = incidence_df["incidence_per_100k"].to_numpy(dtype=float)

xcorr_results = {}
summary_rows = []
for item in PARAMS:
    data = param_data[item["key"]]
    monthly_mat = interpolate_matrix_to_target_time(data["time"], data["mat"], inc_time)
    monthly_median = np.nanmedian(monthly_mat, axis=0)
    lag_df = scan_lags(monthly_median, inc_values)
    best = find_best_lag(lag_df, criterion="abs")
    lag0 = lag_df.loc[lag_df["lag_months"] == 0].iloc[0]
    xcorr_results[item["key"]] = {
        **item,
        "monthly_mat": monthly_mat,
        "monthly_median": monthly_median,
        "lag_df": lag_df,
        "best": best,
        "lag0": lag0,
    }
    summary_rows.append({
        "parameter": item["title"],
        "best_lag_months": int(best["lag_months"]) if best is not None else np.nan,
        "best_spearman_rho": float(best["correlation"]) if best is not None else np.nan,
        "best_p_value": float(best["p_value"]) if best is not None else np.nan,
        "lag0_spearman_rho": float(lag0["correlation"]),
        "lag0_p_value": float(lag0["p_value"]),
        "n_pairs_lag0": int(lag0["n_pairs"]),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_CSV, index=False)
display(summary_df)
print(f"Saved summary: {SUMMARY_CSV}")


In [ ]:

def plot_cross_correlation(xcorr_results):
    all_corr = np.concatenate([res["lag_df"]["correlation"].dropna().to_numpy() for res in xcorr_results.values()])
    ymin = max(-1.0, np.nanmin(all_corr) - 0.08)
    ymax = min(1.0, np.nanmax(all_corr) + 0.08)

    fig, axes = plt.subplots(1, 2, figsize=(9.7, 3.75), sharey=True, constrained_layout=False)
    fig.patch.set_facecolor("white")

    for idx, (ax, item) in enumerate(zip(axes, PARAMS)):
        res = xcorr_results[item["key"]]
        lag_df = res["lag_df"]
        best = res["best"]
        lag0 = res["lag0"]
        color = item["line_color"]

        ax.plot(lag_df["lag_months"], lag_df["correlation"], color="#3F4349", lw=1.25, zorder=2)
        ax.scatter(lag_df["lag_months"], lag_df["correlation"], s=9, color="#3F4349", alpha=0.65, zorder=3)
        ax.axhline(0, color="#A8ADB3", ls="--", lw=0.7, zorder=0)
        ax.axvline(0, color="#A8ADB3", ls="--", lw=0.7, zorder=0)

        if best is not None:
            ax.axvline(best["lag_months"], color=color, lw=0.8, alpha=0.55, zorder=1)
            ax.scatter(best["lag_months"], best["correlation"], s=46, facecolor="#F0C8C8", edgecolor=color, lw=0.8, zorder=5)
            note = (
                f"best lag = {int(best['lag_months'])} mo\n"
                f"rho = {best['correlation']:.2f}\n"
                f"lag 0 rho = {lag0['correlation']:.2f}"
            )
            ax.text(
                0.05,
                0.93,
                note,
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=8.2,
                linespacing=1.25,
                bbox=dict(boxstyle="round,pad=0.28", facecolor="white", edgecolor="#D9DDE2", linewidth=0.65, alpha=0.92),
            )

        ax.set_title(item["title"], fontsize=12.5, pad=7)
        ax.set_xlim(min(LAG_RANGE_MONTHS), max(LAG_RANGE_MONTHS))
        ax.set_ylim(ymin, ymax)
        ax.set_xlabel("Lag (months)", fontsize=8.8)
        if idx == 0:
            ax.set_ylabel("Spearman correlation with incidence", fontsize=8.8)
        ax.grid(axis="y", color="#D9DDE2", lw=0.55, alpha=0.85)
        ax.grid(axis="x", color="#EEF0F2", lw=0.4, alpha=0.45)
        ax.tick_params(axis="both", labelsize=8, colors="#222222", length=3)
        for side in ["top", "right"]:
            ax.spines[side].set_visible(False)
        ax.spines["left"].set_color("#333333")
        ax.spines["bottom"].set_color("#333333")

    fig.suptitle("Lagged cross-correlation between continuous diffusion summaries and global incidence", fontsize=12.8, y=0.99)
    fig.text(0.5, 0.015, "Positive lag means the diffusion parameter leads incidence.", ha="center", va="bottom", fontsize=8, color="#555555")
    plt.subplots_adjust(left=0.075, right=0.985, bottom=0.19, top=0.80, wspace=0.18)
    return fig


fig = plot_cross_correlation(xcorr_results)
fig.savefig(FIG_XCORR_PDF, format="pdf", bbox_inches="tight")
fig.savefig(FIG_XCORR_PNG, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_XCORR_PDF}")
print(f"Saved: {FIG_XCORR_PNG}")
